# Multigroup operator and parametrization




We first create the same circular domain but give some materials a different material (same materials as in the spatial discretization notebooks.)

In [ ]:
import jax
jax.config.update("jax_enable_x64", True)
jax.config.update("jax_platforms", "cpu")
import jax.numpy as jnp
from jax_sbgeom.flux_surfaces.miller_tokamak import MillerTokamak
import jax_sbgeom as jsb
from jax_sn.domain import Domain, CrossSectionsIndexed, CrossSectionData
from jax_sn.solution_domain import BasixLagrangianSimplex
from jax_sn.tools.plotly import plot_domain

tokamak                 = MillerTokamak(rminor = 1.0, rmajor = 1.2, elongation = 0.5, triangularity = 0.2)

def construct_domain_solution_element_quadrature_set(n_s, n_theta):


    mesh_tokamak            = jsb.flux_surfaces.flux_surface_meshing.mesh_poloidal_plane(tokamak, jnp.linspace(0.0,1.0,n_s), 0.0, n_theta)
    # Convert it to a 2D mesh!
    mesh_tokamak           = (jnp.stack([mesh_tokamak[0][:,0], mesh_tokamak[0][:,2]], axis=-1), mesh_tokamak[1])
    water = CrossSectionData(
        total = jnp.array([15.2090, 24.4140]),
        scattering = jnp.array([
            [
                [5.5069, 0.0],
                [6.6227, 11.4080]
            ],
            [
                [1.158567, 0.0],
                [1.1234767, 2.4733]
            ]
        ])
    )
    iron = CrossSectionData(
        total = jnp.array([30.3110, 26.8760]),
        scattering = jnp.array([
            [
                [18.2500, 0.0],
                [6.45294, 21.3570]
            ],
            [
                [4.36, 0.0],
                [-0.003508, 2.3457]
            ]        
        ])
    )
    void = CrossSectionData(
        total = jnp.array([1e-10, 1e-10]),
        scattering = jnp.zeros((2, 2, 2))
    )

    water_idx, iron_idx, void_idx = 0, 1, 2
    cross_section_indices         = jnp.concatenate([jnp.full((mesh_tokamak[1].shape[0]//3,), water_idx), jnp.full((mesh_tokamak[1].shape[0]//3,), iron_idx), jnp.full((mesh_tokamak[1].shape[0] - 2*(mesh_tokamak[1].shape[0]//3),), void_idx)])
    cross_sections_tokamak        = CrossSectionsIndexed(indices = cross_section_indices, cross_section_data = CrossSectionData.from_list([water, iron, void]))


    element_order = 2
    solution_element = BasixLagrangianSimplex(element_order, 2)

    quadrature_order = 3
    from jax_sn.quadrature_set import create_tn_quadrature_set
    quadrature_set  = create_tn_quadrature_set(quadrature_order)
    domain_tokamak                = Domain.from_mesh_and_cross_sections(mesh_tokamak, solution_element.face_template, cross_sections_tokamak)
    return domain_tokamak, quadrature_set, solution_element



n_s     = 4 
n_theta = 7

domain_tokamak, quadrature_set, solution_element = construct_domain_solution_element_quadrature_set(n_s, n_theta)

plot_domain(domain_tokamak, width=900, height=800, material_names = ["water", "iron", "void"])


Creating a multigroup operator again requires picking a strategy. However, to actually solve the multigroup equation:

$$ (\mathbf{I} - \mathbf{M}\mathbf{T}^{-1}\mathbf{H}\mathbf{D}\mathbf{S})[\phi_{g\ell\epsilon i}]=\mathbf{M}\mathbf{T}^{-1}\mathbf{D}\mathbf{H}[q_{g\ell\epsilon i}] $$

there are actually *two* multigroup operators requires: one including scattering:

$$ \text{Multigroup operator}: (\mathbf{I} - \mathbf{M}\mathbf{T}^{-1}\mathbf{H}\mathbf{D}\mathbf{S})$$

and one that does *not* include scattering:

$$ \text{Multigroup chain operator}: \mathbf{M}\mathbf{T}^{-1}\mathbf{D}\mathbf{H}$$

They can both be constructed and have their own strategy types.


In [ ]:
from jax_sn.operators.multi_group_operator import MultiGroupStrategy, MultiGroupChainStrategy

Currently, the main implementations just use a composed strategy that applies a single-group strategy and then maps it over the groups (using a serial scan or a vmap), or in other words:

$$ \text{for all } g: \quad \mathbf{I} - \underbrace{\mathbf{M}_g \mathbf{T}^{-1}_g \mathbf{H}_g \mathbf{D}_g}_{\text{chain}}\underbrace{\mathbf{S}_g}_{\text{gather-scatter}} $$

This requires the chain and gather-scatter operators to have a particular strategy too. 

Constructing a single-group chain strategy is usually done by combining the suboperatorsm, where each one has its own strategy too:

In [ ]:
from jax_sn.operators.single_group_operator import  ComposedSingleGroupChainStrategy
from jax_sn.operators.transport_operator import VmapSweep, VmapForward, VmapAnglesInversion, VmapAnglesForward
from jax_sn.operators.moment_operator import EinsumFluxMoment
from jax_sn.operators.mass_operator import LaxMass
from jax_sn.operators.scatter_operator import BroadcastedSingleGroupGather

sg_chain_strategy = ComposedSingleGroupChainStrategy(
    transport_strategy           = VmapAnglesForward(VmapForward()),
    transport_inversion_strategy = VmapAnglesInversion(VmapSweep()),
    moment_strategy              = EinsumFluxMoment(),
    mass_strategy                = LaxMass(),
    discrete_strategy            = EinsumFluxMoment()
)

sg_gather_strategy = BroadcastedSingleGroupGather()

The two multigroup strategy objects can then be created:

In [ ]:
from jax_sn.operators.multi_group_operator import MultiGroupOperator, MultiGroupChainOperator, MultiGroupChainGatherScanStrategy, MultiGroupChainScanStrategy
mg_chain_gather_strategy = MultiGroupChainGatherScanStrategy(
    single_group_chain_strategy = sg_chain_strategy,
    single_group_gather_scatter_strategy = sg_gather_strategy,    
)

mg_chain_strategy = MultiGroupChainScanStrategy(
    single_group_chain_strategy = sg_chain_strategy
)
print(mg_chain_gather_strategy)


This is somewhat cumbersome, and in practice you'd probably not construct it like this. We just show it here to illustrate how strategies are used internally.

Instead, you could construct one of the defaults:

In [ ]:
from jax_sn.operator_creation.default_strategies import DefaultMultiGroupStrategy

DefaultMultiGroupStrategy.default_multi_group_strategy()

### Parametrization

Constructing the actual operators requires a {py:class}`~jax_sn.operators.operator_data.OperatorData` object, containing both {py:class}`~jax_sn.operators.operator_data.DiscretisationData` and {py:class}`~jax_sn.operators.operator_data.NonDifferentiableData`. The latter includes the topological sort information, which is never actually differentiated (and cannot be).

Here we will show how to manually construct it, and then introduce the parametrization module which builds all data structures in a much easier way.

In [ ]:
from jax_sn.operators.operator_data import OperatorData, DiscretisationData, SpatialData, AngularData, NonDifferentiableData
from jax_sn.solution_domain import SolutionDomain


operator_data = OperatorData(
    DiscretisationData(
        SolutionDomain.from_element_and_domain(solution_element, domain_tokamak).spatial_data, 
        AngularData.from_quadrature_set(quadrature_set, domain_tokamak.cross_sections_indexed.l_max)
    ),
    NonDifferentiableData(
        sweep_data = mg_chain_gather_strategy.default_sweep_data().from_domain_quadrature_set(domain_tokamak, quadrature_set)
    )
)

mg_operator = MultiGroupOperator(operator_data, mg_chain_gather_strategy)

Besides verbose, it's also not optimal: if you just change the cross sections, doing the above again recomputes everything, from topological connections to angular quadrature data.  
Instead, we can use the parametrization module to construct the operator data efficiently from a set of primitives which define the entire operator:

- Vertices
- Connectivity
- Cross sections
- Finite Element
- Quadrature set / angles
- Strategy 
- keyword arguments to non-differentiable data (in some cases needed; maximum levels in sweep data)

and from this, as above, a multigroup operator (or equivalently, any other operator / data) can be constructed. We define a particular `Builder` class, which creates a particular object from this primitives. 

In [ ]:
from jax_sn.operator_creation.builders import MultiGroupBuilder


mg_builder = MultiGroupBuilder(vertices     = domain_tokamak.vertices, 
                  connectivity = domain_tokamak.connectivity,
                  element      = solution_element,
                  cross_sections_indexed = domain_tokamak.cross_sections_indexed,
                  quadrature_set         = quadrature_set,
                  strategy              = mg_chain_gather_strategy,
                  sweep_data_kwargs     = ()
                  )

print("Results in same operator:", jax.tree.all(jax.tree.map(jnp.array_equal, mg_builder.multi_group_operator, mg_operator)))

This still re-computes all information, even if we just change the strategy!


Also, this is not jittable: 

In [ ]:
import equinox as eqx
@eqx.filter_jit
def mg_operator_from_builder(builder: MultiGroupBuilder):
    return builder.multi_group_operator

try:
    mg_operator_from_builder(mg_builder)
except Exception as e:
    print("JIT failed:", e)

Why? Because the element is a Basix element that's not traceable, and so computing the FEM matrices is impossible under JIT.

To solve these problems, there is parametrization module that can construct all operators and data from these primitives. Furthermore, it provides options to freeze or vary only certain primitives, such that only the data and operators that actually depend on the varying primitives are recomputed. 

We can actually visualize this using a graph:



In [ ]:

from jax_sn.tools.plotly import plot_derivation_graph
plot_derivation_graph(mg_builder, width =900, height = 800 )                              # bare structure
#plot_derivation_graph(MultiGroupBuilder.varying("vertices"))          # + what that policy bought


You can see how each primitive is required for certain quantities to be computed. In this case, everything depends on the free primitives.

Freezing the element type (computing the FEM matrices immediately), makes us able to use JIT for the builder, and visualizing the graph, we indeed see that the element is frozen and the fem_matrices and the face template are stored:

In [ ]:
frozen_element_builder = mg_builder.frozen("element")
mg_operator_from_builder(frozen_element_builder)
plot_derivation_graph(frozen_element_builder, width =800, height = 900 )                              # bare structure

This frozen has an important consequence: each builder has a `updated` function, which safely replaces the primitive, *if this would not invalidate your stored data*. In this case, we are free to e.g. update vertices, but *not* the element type, since we froze it!

In [ ]:
frozen_element_builder.updated(vertices = frozen_element_builder.vertices + 0.1)

try: 
    frozen_element_builder.updated(element = BasixLagrangianSimplex(3, 2))
except Exception as e:
    print("Update failed:", e)

This works the same with other primitives. Let's just only vary the vertices:

In [ ]:
plot_derivation_graph(mg_builder.varying("vertices"), width =800, height = 900 )  

All other primitives are now frozen and the farthest edge in the graph (accessible by just the frozen primitives) is stored. The strategy and sweep_data_kwargs are not frozen: varying the vertices means re-creating everything that depends on them anyway; so freezing is not required.


In short, the parametrization module makes safely constructing operators and updating their data more convenient. All the machinery is JIT'd away in any case, so there's no difference compared to hand-rolling operator updates yourself; only tracing time might be increased a bit by the graph machinery.

In the next notebooks we will exclusively use the parametrization module to construct operators and update their data instead of manually constructing them!